# Filter Braxen by language

> "Needs more corrections"

- branch: master
- hidden: false
- categories: [braxen]

In [25]:
BRAXEN_LOCATION = "/Users/joregan/Playing/braxen/dict/braxen-sv.tsv"

In [26]:
from pathlib import Path

BRAXEN_PATH = Path(BRAXEN_LOCATION)

In [27]:
def clean_phones(phones, strip_accent=True):
    boundaries = ["|", "~", "-", "."]
    out_phones = []
    for phone in phones:
        if phone not in boundaries:
            if strip_accent and phone[0] in ["'", '"', ","]:
                phone = phone[1:]
            out_phones.append(phone)
    return out_phones
    

In [28]:
LANG_FIXES = """
playback 376738 eng
playboy 376739 eng
play-off 782183 eng
playa 376734 spa
Halloweens 880274 eng
återvändaren 837944 swe
temporomandibular 718339 swe
# (maybe lat? definitely not eng with that pronunciation)
командитното 831898 bul
Štépnička 869924 cze
Štépničkas 869925 cze
Škoda 733908 cze
Bandsåg 739294 swe
# also 'VB PRT AKT' -> NN UTR SIN IND NOM (?)
Gomez 634927 spa
Gražinytė 877254 lit
Gražinytės 877255 lit
Schouw 926457 dut
Aschehoug 620855 nob
"""

In [29]:
lang_fixes = {}
for line in LANG_FIXES.strip().split("\n"):
    if line.startswith("#"):
        continue
    parts = line.strip().split(" ")
    word, word_id, lang = parts[0], parts[1], parts[2]
    lang_fixes[word_id] = lang

In [30]:
filtered = {}

with open(BRAXEN_LOCATION) as inf:
    for line in inf.readlines():
        if line.startswith("#"):
            continue
        parts = line.strip().split("\t")
        word, phones, pos, lang = parts[0], parts[1], parts[2], parts[3]
        word_id = parts[-1]

        if word_id in lang_fixes:
            lang = lang_fixes[word_id]

        if not lang in filtered:
            filtered[lang] = []
        phones = clean_phones(phones.split(" "), False)
        filtered[lang].append((word, phones, pos, word_id))

In [31]:
filtered.keys()

dict_keys(['swe', 'ara', 'eng', 'dan', 'ger', 'nob', 'fre', 'mix', 'unk', 'dut', 'fin', 'fisa', 'per', 'smi', 'spa', 'heb', 'rus', 'ice', 'por', 'sla', 'afr', 'ita', 'tur', 'tir', 'ind', 'hin', 'hun', 'arm', 'lat', 'gre', 'pol', 'jpn', 'cze', 'asi', 'cat', 'alb', 'gle', 'srp', 'syr', 'aze', 'fic', 'lit', 'chi', 'wel', 'kor', 'kal', 'rum', 'bul', 'ukr', 'yid', 'baq', 'hrv', 'geo', 'kaz', 'rom', 'lav', 'mlt', 'slo', 'fao', 'bos', 'kur', 'gla', 'tib', 'aus', 'som', 'tha', 'est', 'swa', 'ltz', 'nep', 'bur', 'urd', 'haw', 'nno', 'pus', 'uzb', 'vie', 'zul', 'yor', 'san', 'ibo', 'bel', 'mao', 'NN'])

In [32]:
for langname in filtered:
    with open("/tmp/filtered-{}.txt".format(langname), "w") as outf:
        for word, phones, pos, word_id in filtered[langname]:
            outf.write("{}\t{}\n".format(word.lower(), " ".join(phones)))